# Estimating C

In [2]:
%conda init
%conda activate zhiran

no change     /work/home/H.extreme/miniconda3/condabin/conda
no change     /work/home/H.extreme/miniconda3/bin/conda
no change     /work/home/H.extreme/miniconda3/bin/conda-env
no change     /work/home/H.extreme/miniconda3/bin/activate
no change     /work/home/H.extreme/miniconda3/bin/deactivate
no change     /work/home/H.extreme/miniconda3/etc/profile.d/conda.sh
no change     /work/home/H.extreme/miniconda3/etc/fish/conf.d/conda.fish
no change     /work/home/H.extreme/miniconda3/shell/condabin/Conda.psm1
no change     /work/home/H.extreme/miniconda3/shell/condabin/conda-hook.ps1
no change     /work/home/H.extreme/miniconda3/lib/python3.13/site-packages/xontrib/conda.xsh
no change     /work/home/H.extreme/miniconda3/etc/profile.d/conda.csh
no change     /work/home/H.extreme/.bashrc
No action taken.

Note: you may need to restart the kernel to use updated packages.

CondaError: Run 'conda init' before 'conda activate'


Note: you may need to restart the kernel to use updated packages.


In [3]:
import xarray as xr
import netCDF4 as nc

year = '1980'
month = '01'
ds = xr.open_dataset(f"/work5/ERA5_daily_landgrid_only/global/compressed_sfc_fc_daily_{year}{month}.nc4")
ds

<xarray.Dataset> Size: 688MB
Dimensions:   (lgrid: 141780, time: 31)
Coordinates:
  * time      (time) float64 248B 1.98e+07 1.98e+07 ... 1.98e+07 1.98e+07
Dimensions without coordinates: lgrid
Data variables: (12/43)
    lat       (lgrid) float32 567kB ...
    lon       (lgrid) float32 567kB ...
    cin       (time, lgrid) float32 18MB ...
    zust      (time, lgrid) float32 18MB ...
    msror     (time, lgrid) float32 18MB ...
    mssror    (time, lgrid) float32 18MB ...
    ...        ...
    flsr      (time, lgrid) float32 18MB ...
    lat_2     (lgrid) float32 567kB ...
    lon_2     (lgrid) float32 567kB ...
    mcpr      (time, lgrid) float32 18MB ...
    sp        (time, lgrid) float32 18MB ...
    tcc       (time, lgrid) float32 18MB ...
Attributes:
    CDI:          Climate Data Interface version 1.8.2 (http://mpimet.mpg.de/...
    history:      Fri Apr 16 08:48:47 2021: cdo -z zip_1 -f nc4 -O copy compr...
    Conventions:  CF-1.6
    description:  ERA5 daily statistics computed from hourly output on N320 r...
    CDO:          Climate Data Operators version 1.8.2 (http://mpimet.mpg.de/...

In [11]:
from scipy.constants import convert_temperature
from numpy import exp, sqrt, log

def estimate_C(ds):

    # T: air temperature at 2-meter (°C)
    T = convert_temperature(ds['2t'], old_scale='Kelvin', new_scale='Celsius')
    
    # Delta: Slope of the saturation vapor pressure curve at a given air temperature (kPa/°C)
    Delta = ( 2503 * exp((17.27*T)/(T+237.3)) ) / ( (T+237.3)**2 )
    
    # P: surface atmospheric pressure (kPa)
    P = ds['sp'] / 1000 # from Pa to kPa
    
    # gamma: Constant multiplied by surface pressure P
    gamma = 0.000665 * P
    
    # u_2: Wind speed at 2-meter (m/s)
    U_10 = sqrt(ds['10u']**2 + ds['10v']**2) # Magnitude of wind speed at 10 m 
    U_2 = U_10 * (4.87 / (log(67.8*10 - 5.42)))
    
    # KPa / K
    C = Delta + gamma*( 1 + 0.34*U_2 )

    return C
    
C = estimate_C(ds)

# Save array to file
C.to_netcdf(f"C_estimates/C_{year}{month}.nc")

## Calculating for past 45 years (1979-2023)

In [18]:
# Calculate Jan to Dec of 1979 to 2023
years = range(1979, 2024)
months = range(1, 13)

# Counter to track progress
completed = 0
total = len(years)*len(months)

for year in years:
    for month in months:
        
        # Open file for that month
        ds = xr.open_dataset(f"/work5/ERA5_daily_landgrid_only/global/compressed_sfc_fc_daily_{year}{month:02}.nc4")
        
        # Estimate C
        C = estimate_C(ds)

        # TODO: save lat, lon, and give proper variable names
        
        # Save result to .nc file
        C.to_netcdf(f"C_estimates/C_{year}{month:02}.nc")

        completed += 1
        print(f"[{completed}/{total}] Data from {year}-{month:02} was processed correctly")

print("Done! All estimates were calculated and saved correctly.")

[1/540] Data from 1979-01 was processed correctly
[2/540] Data from 1979-02 was processed correctly
[3/540] Data from 1979-03 was processed correctly
[4/540] Data from 1979-04 was processed correctly
[5/540] Data from 1979-05 was processed correctly
[6/540] Data from 1979-06 was processed correctly
[7/540] Data from 1979-07 was processed correctly
[8/540] Data from 1979-08 was processed correctly
[9/540] Data from 1979-09 was processed correctly
[10/540] Data from 1979-10 was processed correctly
[11/540] Data from 1979-11 was processed correctly
[12/540] Data from 1979-12 was processed correctly
[13/540] Data from 1980-01 was processed correctly
[14/540] Data from 1980-02 was processed correctly
[15/540] Data from 1980-03 was processed correctly
[16/540] Data from 1980-04 was processed correctly
[17/540] Data from 1980-05 was processed correctly
[18/540] Data from 1980-06 was processed correctly
[19/540] Data from 1980-07 was processed correctly
[20/540] Data from 1980-08 was processed